In [ ]:
import numpy as np
import time
import cv2
import os
from sdlarch_rl.utils.stf6_imitation import STF6Env
from IPython.display import Audio
from stable_baselines3 import PPO
# from sbx import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path
from sdlarch_rl.utils.utils import get_last_index, GenericCNN
from stable_baselines3.common.policies import ActorCriticPolicy
import sys
import gymnasium as gym

import logging
logging.basicConfig(level=logging.DEBUG)


# NUM_ENV = 6
NUM_ENV = 1
SAVE_DIR="./model-sf6"
TENSORBOARD="./tensorboard-sf6"
TOTAL_TIMESTEP_NUMB = 50_000_000
CHECK_FREQ_NUMB = 5_000
SAVE_FREQ = CHECK_FREQ_NUMB
EVAL_FREQU=CHECK_FREQ_NUMB*2
MAX_STEPS= 4_000
LEARNING_RATING= 1e-6 # 1e-5# 5e-5 # 2e-4
EVALS=10
clip_range=0.1

#ENT_COEF = 0.001
ENT_COEF = 0 # 0.01 # 0.00001 # 0.001 # 0.01 # 0.001 # 0.00001
n_steps= 8192 # 2048 # 8192 # 4096 # 2048
# batch_size=64 * NUM_ENV
batch_size= 1024 * NUM_ENV # 128 * NUM_ENV

SAVE_DIR = Path(SAVE_DIR)

def linear_schedule(initial_lr):
    def schedule(progress_remaining):
        return progress_remaining * initial_lr
    return schedule

def make_env():
    def _init():
        env = STF6Env()
        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=1)
        env = TimeLimit(env, max_steps=MAX_STEPS)
        return env
    return _init

# env = make_vec_env(make_env(), n_envs=NUM_ENV, vec_env_cls=SubprocVecEnv)
env = make_vec_env(make_env(), n_envs=NUM_ENV, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4)

latest_model_path = get_latest_model(SAVE_DIR)

# eval_env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
# eval_env = VecFrameStack(eval_env, 4)


def get_previous_best_reward(log_dir):
    """Better reward"""
    eval_logs_dir = Path(log_dir) / "eval_logs"
    
    if not eval_logs_dir.exists():
        print("Directory of eval_logs not founded")
        return -np.inf
    
    npz_files = list(eval_logs_dir.glob("*.npz"))
    if not npz_files:
        print("No .npz files found")
        return -np.inf
    
    best_mean_reward = -np.inf
    
    for npz_file in sorted(npz_files):
        try:
            data = np.load(npz_file, allow_pickle=True)
            
            if 'results' in data:
                results = data['results']
                
                for eval_idx in range(results.shape[0]):
                    episode_rewards = results[eval_idx]
                    
                    mean_reward = np.mean(episode_rewards)
                    
                    if mean_reward > best_mean_reward:
                        best_mean_reward = mean_reward
                        
        except Exception as e:
            print(f"Error on ready {npz_file.name}: {e}")
            continue
    
    if best_mean_reward > -np.inf:
        print(f"Better rewarnd mean (mean of n_eval_episodes episodes): {best_mean_reward:.4f}")
        return best_mean_reward
    
    return -np.inf

previous_best_reward = get_previous_best_reward(SAVE_DIR)

class PersistentEvalCallback(EvalCallback):
    """
    EvalCallback preserves the history and saves it after each evaluation.
    """
    def __init__(self, *args, initial_best=-np.inf, **kwargs):
        super().__init__(*args, **kwargs)
        
        self.initial_best = initial_best
        self.has_logged_goal = False
        self.log_file = Path(self.log_path + ".npz")
        
        if initial_best > -np.inf:
            self.best_mean_reward = initial_best
            print(f"\nEvalCallback using history data: {self.best_mean_reward:.4f}")

    def _on_step(self) -> bool:
        if not hasattr(self, 'best_mean_reward_initialized'):
            if self.initial_best > -np.inf:
                self.best_mean_reward = self.initial_best
            self.best_mean_reward_initialized = True

        if self.n_calls == 1 and self.initial_best > -np.inf and not self.has_logged_goal:
            print(f"Objetive: {self.initial_best:.4f}")
            self.has_logged_goal = True
        
        result = super()._on_step()
        
        if hasattr(self, 'last_eval_timestep') and self.model.num_timesteps > self.last_eval_timestep:
            self._save_evaluation_data()
        
        return result
    
    def _on_evaluation(self, locals_, globals_):
        """Call after any eval - stores the timestep"""
        print("_on_evaluation")
        super()._on_evaluation(locals_, globals_)
        self.last_eval_timestep = self.model.num_timesteps
    
    def _save_evaluation_data(self):
        """Save data from eval to file .npz"""
        print("_save_evaluation_data")
        if len(self.timesteps) == 0:
            return
        
        try:
            last_idx = -1
            new_timestep = self.timesteps[last_idx]
            new_result = self.results[last_idx]
            new_ep_length = self.ep_lengths[last_idx]
            
            if self.log_file.exists():
                existing_data = np.load(self.log_file, allow_pickle=True)
                
                if new_timestep in existing_data['timesteps']:
                    print(f"Eval timestep {new_timestep} already exist. Skipping.")
                    return
                
                combined_timesteps = np.concatenate([existing_data['timesteps'], [new_timestep]])
                combined_results = np.vstack([existing_data['results'], [new_result]])
                combined_ep_lengths = np.vstack([existing_data['ep_lengths'], [new_ep_length]])
                
            else:
                combined_timesteps = np.array([new_timestep])
                combined_results = np.array([new_result])
                combined_ep_lengths = np.array([new_ep_length])
            
            np.savez(
                self.log_file,
                timesteps=combined_timesteps,
                results=combined_results,
                ep_lengths=combined_ep_lengths
            )
            
            total = len(combined_timesteps)
            print(f"Saved: eval {total} (timestep: {new_timestep:,})")
            
            if len(self.timesteps) > 0:
                self.timesteps.pop(last_idx)
                self.results.pop(last_idx)
                self.ep_lengths.pop(last_idx)
            
        except Exception as e:
            print(f"Error on save eval: {e}")
            import traceback
            traceback.print_exc()


workaround_value = 0

if previous_best_reward > workaround_value:
    print(f"Using better historical: {previous_best_reward:.4f}")
else:
    print(f"Using default historical workaround:{workaround_value}")
    previous_best_reward = workaround_value

# eval_callback = PersistentEvalCallback(
#     eval_env,
#     best_model_save_path=SAVE_DIR,
#     log_path=str(SAVE_DIR / "eval_logs"),
#     eval_freq=EVAL_FREQU,
#     deterministic=True,
#     render=False,
#     n_eval_episodes=EVALS,
#     initial_best=previous_best_reward
# )

if latest_model_path:
    print(f"Loading existent model: {latest_model_path}")
    model = PPO.load(
        str(latest_model_path), 
        env=env, 
        verbose=0, 
        tensorboard_log=TENSORBOARD, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        learning_rate=linear_schedule(LEARNING_RATING),
        batch_size=batch_size,
        clip_range=clip_range,
    )
    
else:
    print("None finded, starting from zero.")

    train_path = 'imitation-sf6/'

    last_index_imitation = int(get_last_index(train_path, "bc_policy", "zip"))
    latest_model_path = train_path + f"bc_policy{last_index_imitation}.zip"

    print("loading from: " + str(latest_model_path))

    a2c = ActorCriticPolicy.load(
        str(latest_model_path),
    )
    
    policy_kwargs = dict(
        net_arch=dict(pi=[256, 256], vf=[256, 256]), 
        features_extractor_class=GenericCNN
    )
    
    model = PPO(
        policy=a2c.__class__,
        env=env,
        policy_kwargs=policy_kwargs,
        verbose=0, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
        tensorboard_log=TENSORBOARD, 
        learning_rate=linear_schedule(LEARNING_RATING),
        clip_range=clip_range,
    )

    model.policy.load_state_dict(a2c.state_dict(), strict=False)
    
    # model = PPO("CnnPolicy", 
    # # model = RecurrentPPO('CnnLstmPolicy',
    #     env, 
    #     verbose=0, 
    #     # policy_kwargs=policy_kwargs, 
    #     ent_coef =ENT_COEF,
    #     n_steps=n_steps,
    #     batch_size=batch_size,
    #     tensorboard_log=TENSORBOARD, 
    #     learning_rate=linear_schedule(LEARNING_RATING)
    # )


checkpoint_callback=TrainAndLoggingCallback(check_freq=CHECK_FREQ_NUMB, save_path=SAVE_DIR, save_freq=SAVE_FREQ, model=model)
# callback = CallbackList([checkpoint_callback, eval_callback])
callback = CallbackList([checkpoint_callback])

model.learn(total_timesteps=TOTAL_TIMESTEP_NUMB, reset_num_timesteps=False, callback=callback)
model.save("final_sf6")

env.close()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


Directory of eval_logs not founded
Using default historical workaround:0
Loading existent model: model-gt3\best_model_150000


D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:449: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  th_object = th.load(file_content, map_location=device

Done Rewards Step Cnt: 4001
Model saved in: model-gt3\best_model_155000
Done Rewards Step Cnt: 1995
Done Rewards Step Cnt: 2155
Model saved in: model-gt3\best_model_160000
Done Rewards Step Cnt: 1972
Done Rewards Step Cnt: 3257
Done Rewards Step Cnt: 1030
Model saved in: model-gt3\best_model_165000
Done Rewards Step Cnt: 613
Done Rewards Step Cnt: 3300
Model saved in: model-gt3\best_model_170000
Done Rewards Step Cnt: 4001
Model saved in: model-gt3\best_model_175000
Done Rewards Step Cnt: 4001
Done Rewards Step Cnt: 2553
Model saved in: model-gt3\best_model_180000
Done Rewards Step Cnt: 3453
Done Rewards Step Cnt: 2192
Model saved in: model-gt3\best_model_185000
Done Rewards Step Cnt: 1998
Done Rewards Step Cnt: 1112
Done Rewards Step Cnt: 1094
Model saved in: model-gt3\best_model_190000
Done Rewards Step Cnt: 3382
Done Rewards Step Cnt: 1194
Done Rewards Step Cnt: 1111
Model saved in: model-gt3\best_model_195000
Done Rewards Step Cnt: 726
